# Gemini Enterprise Usage
Simple and quick usage metrics for GE

![ge_usage_architecture.png](ge_usage_architecture.png)

## Modelado de tabla de user events con bigframes


**Reasoning**:
Load the user events data from BigQuery using bigframes.pandas to begin the aggregation subtask.



In [9]:
import bigframes.pandas as bpd

**Reasoning**:
Load the identified user activity table from the user events dataset using BigFrames and inspect its schema to identify the required columns for aggregation.



In [28]:
gcp_project = "lgbaeza-202310"

bq_table_assistant = f"{gcp_project}.gemini_e_analytics.discoveryengine_googleapis_com_gemini_enterprise_user_activity"
target_table_assistant = f"{gcp_project}.gemini_e_analytics.assistant_model"

bq_table_user_events = f"{gcp_project}.gemini_e_analytics_userev.discoveryengine_googleapis_com_gemini_enterprise_user_activity"
target_table_user = f"{gcp_project}.gemini_e_analytics_userev.user_events_model"

In [29]:
df_user_events = bpd.read_gbq(bq_table_user_events)

/usr/local/lib/python3.12/dist-packages/bigframes/core/logging/log_adapter.py:183: TimeTravelCacheWarning: Reading cached table from 2026-08-13 16:18:16.373262+00:00 to avoid
incompatibilies with previous reads of this table. To read the latest
version, set `use_cache=False` or close the current session with
Session.close() or bigframes.pandas.close_session().
  return method(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bigframes/session/_io/bigquery/read_gbq_table.py:377: DefaultIndexWarning: Table 'lgbaeza-
202310.gemini_e_analytics_userev.discoveryengine_googleapis_com_gemini
_enterprise_user_activity' is clustered and/or partitioned, but
BigQuery DataFrames was not able to find a suitable index. To avoid
this warning, set at least one of: `index_col` or `filters`.
  warnings.warn(msg, category=bfe.DefaultIndexWarning)


In [30]:
df_user_events['day'] = df_user_events['timestamp'].dt.date

agg_user_events = df_user_events.groupby(['day', 'logName']).agg(
    total_events=('insertId', 'count'),
    distinct_principals=('insertId', 'nunique') # Placeholder until principal_id is confirmed
).reset_index()

In [31]:
# Extract nested fields from the jsonPayload struct
df_user_events['event_type'] = df_user_events['jsonPayload'].struct.field('request').struct.field('userevent').struct.field('eventtype')
df_user_events['principal_id'] = df_user_events['jsonPayload'].struct.field('useriamprincipal')

# Ensure the day is extracted (already done in step 3, but redefined for safety in this block)
df_user_events['day'] = df_user_events['timestamp'].dt.date

# Perform the final aggregation as requested
# Aggregate by day and event type to calculate total event counts and distinct principal_id counts
user_events_model = df_user_events.groupby(['day', 'event_type']).agg(
    total_event_count=('insertId', 'count'),
    distinct_principal_count=('principal_id', 'nunique')
).reset_index()

## Modelado de tabla de assistant events con bigframes


In [32]:
df_assistant_events = bpd.read_gbq(bq_table_assistant)

/usr/local/lib/python3.12/dist-packages/bigframes/core/logging/log_adapter.py:183: TimeTravelCacheWarning: Reading cached table from 2026-08-13 16:21:48.903243+00:00 to avoid
incompatibilies with previous reads of this table. To read the latest
version, set `use_cache=False` or close the current session with
Session.close() or bigframes.pandas.close_session().
  return method(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bigframes/session/_io/bigquery/read_gbq_table.py:377: DefaultIndexWarning: Table 'lgbaeza-
202310.gemini_e_analytics.discoveryengine_googleapis_com_gemini_enterp
rise_user_activity' is clustered and/or partitioned, but BigQuery
DataFrames was not able to find a suitable index. To avoid this
warning, set at least one of: `index_col` or `filters`.
  warnings.warn(msg, category=bfe.DefaultIndexWarning)


In [34]:
df_assistant_events['day'] = df_assistant_events['timestamp'].dt.date
df_assistant_events['agent_id'] = df_assistant_events['jsonPayload'].struct.field('response').struct.field('agentinfo').struct.field('agent')
df_assistant_events['agent_name'] = df_assistant_events['jsonPayload'].struct.field('response').struct.field('agentinfo').struct.field('displayname')
df_assistant_events['principal_id'] = df_assistant_events['jsonPayload'].struct.field('useriamprincipal')

# Perform aggregation as per requirements
assistant_events_model = df_assistant_events.groupby(['day', 'agent_name', 'agent_id']).agg(
    total_request_count=('insertId', 'count'),
    distinct_principal_count=('principal_id', 'nunique')
).reset_index()

## Materializar modelos agregados


In [35]:
user_events_model.to_gbq(target_table_user, if_exists='replace')
print(f"Table {target_table_user} created successfully.")

Table lgbaeza-202310.gemini_e_analytics_userev.user_events_model created successfully.


In [36]:
assistant_events_model.to_gbq(target_table_assistant, if_exists='replace')
print(f"Table {target_table_assistant} created successfully.")

Table lgbaeza-202310.gemini_e_analytics.assistant_model created successfully.


## Definición de esquemas finales


#### 1. Modelo de Eventos de Usuario (`user_events_model`)
Este modelo permite analizar la actividad diaria de los usuarios según el tipo de acción realizada.

| Campo | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| `day` | date32[day][pyarrow] | Fecha de la actividad extraída del timestamp original. |
| `event_type` | string[pyarrow] | Categoría de la interacción (ej. 'add-feedback'). |
| `total_event_count` | Int64 | Volumen total de eventos registrados para ese día y tipo. |
| `distinct_principal_count` | Int64 | Número de usuarios únicos (principals) que generaron dichos eventos. |

#### 2. Modelo de Eventos del Asistente (`assistant_events_model`)
Este modelo está orientado al análisis del rendimiento y adopción de agentes específicos.

| Campo | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| `day` | date32[day][pyarrow] | Fecha de la petición al asistente. |
| `agent_name` | string[pyarrow] | Nombre legible del agente del asistente. |
| `agent_id` | string[pyarrow] | Identificador único del recurso del agente. |
| `total_request_count` | Int64 | Cantidad total de peticiones procesadas por el agente en el día. |
| `distinct_principal_count` | Int64 | Cantidad de usuarios únicos que interactuaron con el agente. |

### Valor Analítico
Estos modelos proporcionan una base sólida para:
- **Análisis de Alcance:** Evaluar cuántos usuarios únicos utilizan el sistema diariamente.
- **Volumen de Interacción:** Identificar picos de demanda y los tipos de eventos o agentes más utilizados.
- **Eficiencia del Agente:** Comparar el uso entre diferentes agentes para optimizar recursos.

![ge_usage.png](ge_usage.png)